In [ ]:
#Imports and path setup
import sys
import os
sys.path.insert(0, os.path.abspath('../src'))

import io
import ipywidgets as widgets
import pandas as pd
import numpy as np
import plotly.graph_objects as go

from optimization import MeanVariance, QEQW, LeastSquares, LAD
from optimization_data import OptimizationData
from constraints import Constraints

In [6]:
#File upload
uploader = widgets.FileUpload(
    description='Upload CSV',
    accept='.csv',
    multiple=False
)
uploader

FileUpload(value=(), accept='.csv', description='Upload CSV')

In [15]:
#Load and preview data
try:
    uploaded_file = uploader.value[0]
except IndexError:
    print("No file uploaded. Please upload a CSV file to proceed.")
    raise
content = uploaded_file['content']
df = pd.read_csv(io.BytesIO(content), index_col=0, parse_dates=True)
print(f"Loaded data: {df.shape[0]} rows, {df.shape[1]} assets")
df.head()

Loaded data: 6338 rows, 1 assets


,NDDLWI
Index,
01-01-1999,-0.000072
04-01-1999,0.008686
05-01-1999,0.009618
06-01-1999,0.020803
07-01-1999,-0.003080


In [13]:
#Optimisation controls
strategy_dropdown = widgets.Dropdown(
    options=['Least Squares', 'Weighted Least Squares', 'LAD', 'Mean Variance'],
    value='Least Squares',
    description='Strategy:',
    style={'description_width': 'initial'}
)

n_assets_slider = widgets.IntSlider(
    value=10,
    min=2,
    max=24,
    step=1,
    description='Number of Assets:',
    style={'description_width': 'initial'}
)

risk_aversion_slider = widgets.FloatSlider(
    value=1.0,
    min=0.1,
    max=5.0,
    step=0.1,
    description='Risk Aversion:',
    style={'description_width': 'initial'}
)

controls = widgets.VBox([strategy_dropdown, n_assets_slider, risk_aversion_slider])
controls

In [16]:
#Run optimisation and visualise results

STRATEGY_MAP = {
    'Least Squares': LeastSquares,
    'LAD': LAD,
    'Mean Variance': MeanVariance,
    'Weighted Least Squares': None  # requires extra params, coming soon
}

def plot_weights(weights_dict):
    """Render an interactive Plotly pie chart of portfolio weights."""
    filtered = {k: v for k, v in weights_dict.items() if v > 0.001}
    fig = go.Figure(data=[go.Pie(
        labels=list(filtered.keys()),
        values=list(filtered.values()),
        hole=0.3,
        textinfo='label+percent'
    )])
    fig.update_layout(title='Portfolio Allocation', showlegend=True)
    fig.show()

def on_run_clicked(b):
    with output:
        output.clear_output()

        strategy_name = strategy_dropdown.value
        StrategyClass = STRATEGY_MAP.get(strategy_name)

        if StrategyClass is None:
            print(f"'{strategy_name}' is not yet supported.")
            return

        print(f"Running {strategy_name} optimisation...")

        n = n_assets_slider.value
        selection = list(df.columns[:n])

        opt_data = OptimizationData(
            return_series=df[selection].iloc[1:].astype(float),
            bm_series=df.iloc[1:, 0].astype(float)
        )

        constraints = Constraints(selection=selection)
        constraints.add_budget()
        constraints.add_box(box_type='LongOnly')

        if strategy_name == 'Mean Variance':
            model = StrategyClass(risk_aversion=risk_aversion_slider.value, constraints=constraints)
        else:
            model = StrategyClass(constraints=constraints)

        model.set_objective(opt_data)
        model.solve()

        weights = model.results['weights']
        print("\nPortfolio Weights:")
        for asset, w in weights.items():
            print(f"  {asset}: {w:.4f}")

        plot_weights(weights)

run_button = widgets.Button(
    description='Run Optimisation',
    button_style='success',
    icon='check'
)
output = widgets.Output()
run_button.on_click(on_run_clicked)
widgets.VBox([run_button, output])